# Speculative RAG as a reranker and explainer on collaborative filtering

A faithful implementation of **Speculative RAG** (Wang et al., 2024, [arXiv:2407.08223](https://arxiv.org/abs/2407.08223)) for movie recommendations, built on top of collaborative-filtering candidates.

**Pipeline:** MovieLens ratings → CF top-50 candidates → plot evidence from Wikipedia → K-means groups → a 7B drafter writes *(rationale, ranking)* for each group in parallel → a 14B verifier scores each draft with the paper's probabilities → the best draft wins.

**Evidence that it works:** held-out future ratings from real MovieLens users, an ablation table against strong CF baselines and a single Gemini call, NLI-checked grounding of the explanations, and measured latency and cost.

### Kaggle settings
- Accelerator **GPU T4 ×2**, Internet **on**.
- Add-ons → Secrets: `GEMINI_API_KEY` (required for §9 and §10) and `HF_TOKEN` (optional; the default models are not gated).
- **First run:** §1 downloads and parses the data (about 30–60 min, needs internet). Save `/kaggle/working/data` as a private Kaggle dataset and attach it, and later runs skip the build.
- **Run 2:** attach one dataset holding `data/*.parquet`, `chunk_emb.npy` and run 1's `results/` files. The notebook finds them anywhere under `/kaggle/input`, reuses run 1's LLM outputs as the shuffled-order control arm, and only computes the new CF-order arm (about 3 h instead of 5–6 h).
- Every expensive stage writes to `/kaggle/working/results`, and LLM stages resume from a partial file after a crash.

### What run 1 found, and what run 2 tests
Run 1 showed the candidates in a random order. Every LLM system then scored **below** CF alone (NDCG@10 0.104), and Spec RAG scored lowest (0.057). Diagnostics on its drafts:
- The model never saw the strongest signal. EASE's top 10 hold 4× more test positives than its ranks 41–50.
- The 7B drafter took 52% of its picks from the first 10 titles listed; with no position bias that would be 20%.
- The verifier's choice was no better than a random draft (0.057 vs 0.059), although the best of the 5 drafts reached 0.110.

**Run 2 changes one thing:** a second arm lists the candidates **in CF order, numbered**, and the prompt says where the order comes from. Users, evidence, subsets, models, verifier and selection rule are identical, so the paired difference between the arms isolates the effect of showing the order.

## §0 Setup and smoke test

In [ ]:
# pip may warn that preinstalled Kaggle packages (grpcio-status, google-adk, ...) want older protobuf/opentelemetry.
# Those packages are never imported here, so the warnings are filtered out; the last line confirms the install.
!pip install -q vllm==0.32.0 implicit mwparserfromhell google-genai sentence-transformers httpx 2>&1 | grep -vE "requires .* but you have|dependency resolver|^$" | tail -n 5
# vLLM upgrades torch, but Kaggle's preinstalled torchaudio stays built for the old CUDA and then crashes any
# transformers import. Nothing here uses audio, so remove it.
!pip uninstall -y -q torchaudio
# import what the vLLM server imports at startup, so a broken environment fails here, not inside the server
!python -c "import torch, torchvision, vllm; from vllm.config import ModelConfig; from transformers import AutoTokenizer; print('OK  vllm', vllm.__version__, '| torch', torch.__version__, '| torchvision', torchvision.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_arch_list())"

In [ ]:
import os, sys, json, time, re, gc, random, asyncio, subprocess, zipfile, urllib.parse, shutil
from pathlib import Path

os.environ["USE_TF"] = "0"  # keep transformers away from Kaggle's TensorFlow, which breaks on the newer protobuf

import numpy as np
import pandas as pd
import scipy.sparse as sp
import requests, httpx
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("./work")
RES = WORK / "results"
for p in (WORK / "data", RES):
    p.mkdir(parents=True, exist_ok=True)

def find_input(name):
    """First file called `name` under /kaggle/input. Kaggle's mount layout varies, so search instead of hardcoding."""
    root = Path("/kaggle/input")
    return next(root.rglob(name), None) if root.exists() else None

_movies = find_input("movies.parquet")  # attach §1's output as a dataset to skip the build
DATA = _movies.parent if _movies else WORK / "data"

# Run 1's LLM outputs, if attached, are the shuffled-order control arm: copied in, the resumable runners skip them
PREV = find_input("drafts_base.jsonl")
PREV = PREV.parent if PREV else None
if PREV:
    for f in ["single_base.jsonl", "drafts_base.jsonl", "verify_base.jsonl", "gemini.jsonl", "walltime_base.json", "latency_base.csv"]:
        if (PREV / f).exists() and not (RES / f).exists():
            shutil.copy(PREV / f, RES / f)

CFG = dict(
    # data and splits
    min_user_ratings=20, holdout=5, pos_threshold=4.0, dislike_threshold=2.0,
    min_item_train_ratings=50, chunk_words=200, min_plot_chars=300,
    # evaluation
    # LLM stages are sized to measured T4 throughput: ~500 prompt tokens/s per GPU for the 7B AWQ drafter
    topk=10, n_cand=50, n_eval_users=500, n_boot=1000, n_latency_users=30,
    # Speculative RAG. The paper retrieves 10 docs and uses k=2 per subset for short-answer QA; ranking 50
    # candidates gets k=3. k and m are fixed up front, not tuned on test.
    n_docs=10, k_clusters=3, m_drafts=5,
    n_liked_in_query=15, n_disliked_in_query=5, n_liked_evidence=5, rocchio_beta=0.5,
    # models
    embed_model="BAAI/bge-m3",
    nli_model="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    drafter_model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    verifier_model="Qwen/Qwen2.5-14B-Instruct-AWQ",
    gemini_model="gemini-flash-latest",  # alias for the current Flash model; the exact id is logged with each call
    # USD per 1M tokens: copy from https://ai.google.dev/gemini-api/docs/pricing for the model above
    gemini_usd_per_m_in=None, gemini_usd_per_m_out=None,
    # §10: LoRA adapter produced by the companion QLoRA notebook (attach it as a dataset)
    lora_path="/kaggle/input/specrag-drafter-lora/drafter_lora",
    n_teacher=2000,
)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

GEMINI_API_KEY = secret("GEMINI_API_KEY")
HF_TOKEN = secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("data dir:", DATA, "| run 1 outputs:", PREV, "| gemini key:", bool(GEMINI_API_KEY), "| hf token:", bool(HF_TOKEN))

In [ ]:
import torch, vllm, transformers

DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
env = dict(torch=torch.__version__, vllm=vllm.__version__, transformers=transformers.__version__,
           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU{i} {env['gpus'][i]}: {free/2**30:.1f} / {total/2**30:.1f} GiB free")
(RES / "env.json").write_text(json.dumps(env, indent=2))
print(env)

vLLM runs as a separate server process for each GPU, which is stopped after use: GPU memory comes back reliably only when the process exits. T4s have no bf16, so the servers run in fp16.

In [ ]:
class VLLMServer:
    def __init__(self, model, gpu, port, name, args=(), served=None):
        self.model, self.gpu, self.port, self.name, self.args = model, gpu, port, name, list(args)
        self.served = served or name
        self.proc = None
        self.url = f"http://localhost:{port}"

    def start(self, timeout=1800):
        cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", self.model,
               "--served-model-name", self.served, "--port", str(self.port), "--dtype", "half",
               "--seed", str(SEED), "--max-logprobs", "20", *self.args]
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(self.gpu)}
        self.log_path = RES / f"vllm_{self.name}.log"
        self.proc = subprocess.Popen(cmd, env=env, stdout=self.log_path.open("w"), stderr=subprocess.STDOUT)
        t0 = time.time()
        while time.time() - t0 < timeout:
            if self.proc.poll() is not None:
                raise RuntimeError(f"vLLM exited; last log lines:\n{self.tail()}")
            try:
                if requests.get(f"{self.url}/health", timeout=2).ok:
                    print(f"{self.name} ready on GPU{self.gpu} after {time.time()-t0:.0f}s")
                    return self
            except requests.RequestException:
                pass
            time.sleep(5)
        self.stop()
        raise TimeoutError(f"{self.name} not ready after {timeout}s:\n{self.tail()}")

    def tail(self, n=15):
        """Root-cause lines (errors usually sit well above the final traceback) plus the end of the log."""
        lines = self.log_path.read_text().splitlines()
        causes = [l for l in lines if re.search(r"Error|error:|out of memory|KV cache|exceeds|insufficient", l)]
        return "\n".join(["-- likely cause --", *causes[-12:], "-- end of log --", *lines[-n:]])

    def stop(self):
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(60)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        self.proc = None

# 16 concurrent drafts fit the drafter's KV cache without preemption (each prompt+output is ~5k tokens at most)
DRAFTER_ARGS = ["--max-model-len", "6144", "--gpu-memory-utilization", "0.90", "--max-num-seqs", "16"]
# 14B AWQ weights take 9.4 of the T4's 15.6 GB. Prompt logprobs materialise full-vocabulary (152k) scores for every
# prompt token in a step, so cap tokens per step and leave headroom outside vLLM's budget for them. The verifier only
# prefills, so eager mode (no CUDA graphs) costs little.
VERIFIER_ARGS = ["--max-model-len", "3072", "--gpu-memory-utilization", "0.85", "--max-num-seqs", "8",
                 "--max-num-batched-tokens", "1024", "--enforce-eager"]
GPUS = [0, 1]

def start_pool(model, served, args, gpus, base_port):
    """One server per GPU. If any fails to start, the ones already running are stopped."""
    pool = []
    try:
        for k, gpu in enumerate(gpus):
            pool.append(VLLMServer(model, gpu=gpu, port=base_port + k, name=f"{served}_gpu{gpu}", args=args, served=served).start())
    except Exception:
        stop_pool(pool)
        raise
    return pool

def stop_pool(pool):
    for srv in pool:
        srv.stop()

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

**Smoke test.** Before anything long runs, check that this vLLM version works on a T4 with an AWQ model and supports both features the pipeline needs: JSON-schema output with per-token logprobs (drafter), and prompt logprobs on a token-id prompt (verifier). If this cell fails, the log tail shows why. Pin the vLLM version that passes in the install cell.

In [ ]:
from transformers import AutoTokenizer

smoke = VLLMServer(CFG["drafter_model"], gpu=0, port=8000, name="drafter_smoke", args=DRAFTER_ARGS, served="drafter").start()
try:
    schema = {"type": "object", "additionalProperties": False, "required": ["rationale", "ranking"],
              "properties": {"rationale": {"type": "string"},
                             "ranking": {"type": "array", "minItems": 2, "maxItems": 2,
                                         "items": {"type": "string", "enum": ["Alien (1979)", "Heat (1995)", "Up (2009)"]}}}}
    r = httpx.post(f"{smoke.url}/v1/chat/completions", timeout=300, json={
        "model": "drafter", "temperature": 0, "max_tokens": 200, "logprobs": True,
        "messages": [{"role": "user", "content": "A user loved Aliens (1986). Rank two of the options for them."}],
        "response_format": {"type": "json_schema", "json_schema": {"name": "draft", "schema": schema}}}).json()
    content = r["choices"][0]["message"]["content"]
    print(content)
    assert json.loads(content)["ranking"][0] in schema["properties"]["ranking"]["items"]["enum"]
    assert r["choices"][0]["logprobs"]["content"], "no generated-token logprobs"

    tok = AutoTokenizer.from_pretrained(CFG["drafter_model"])
    ids = tok(tok.apply_chat_template([{"role": "user", "content": "Say yes."}], add_generation_prompt=True, tokenize=False),
              add_special_tokens=False)["input_ids"]
    r2 = httpx.post(f"{smoke.url}/v1/completions", timeout=300, json={
        "model": "drafter", "prompt": ids, "max_tokens": 1, "temperature": 0, "logprobs": 20, "prompt_logprobs": 0}).json()
    assert len(r2["choices"][0]["prompt_logprobs"]) == len(ids), "prompt logprobs misaligned"
    print("top next tokens:", list(r2["choices"][0]["logprobs"]["top_logprobs"][0].items())[:5])
    print("smoke test passed with vllm", vllm.__version__)
finally:
    smoke.stop()

## §1 Data build (cached)

Raw sources only: MovieLens 25M rating events → `links.csv` (IMDb ids) → Wikidata (P345 → English Wikipedia article) → the article's **Plot** section from the MediaWiki API, parsed with `mwparserfromhell`. Movies with fewer than `min_item_train_ratings` ratings in total are skipped here, because they can't make the final catalog anyway.

In [ ]:
import mwparserfromhell

UA = {"User-Agent": "SpecRAG-MovieRecs/2.0 (research notebook; https://github.com/dhaaivat/SpecRAGvsCFRecommender)"}
WIKI_API = "https://en.wikipedia.org/w/api.php"
PLOT_HEADINGS = r"^\s*(plot|plot summary|synopsis|premise|story|summary)\s*$"

def http_json(url, method="GET", retries=6, **kw):
    for attempt in range(retries):
        try:
            r = requests.request(method, url, headers=UA, timeout=90, **kw)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(float(r.headers.get("Retry-After", 2 ** attempt)))
                continue
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError(f"gave up on {url}")

def wikidata_enwiki_titles(imdb_ids, batch=200):
    """IMDb id -> English Wikipedia title, via Wikidata property P345."""
    out = {}
    for i in tqdm(range(0, len(imdb_ids), batch), desc="wikidata"):
        values = " ".join(f'"{x}"' for x in imdb_ids[i:i + batch])
        query = f"""SELECT ?imdb ?article WHERE {{
          VALUES ?imdb {{ {values} }}
          ?item wdt:P345 ?imdb .
          ?article schema:about ?item ; schema:isPartOf <https://en.wikipedia.org/> .
        }}"""
        js = http_json("https://query.wikidata.org/sparql", method="POST",
                       data={"query": query, "format": "json"})
        for b in js["results"]["bindings"]:
            title = urllib.parse.unquote(b["article"]["value"].rsplit("/wiki/", 1)[1]).replace("_", " ")
            out.setdefault(b["imdb"]["value"], title)
        time.sleep(1)
    return out

def fetch_wikitext(titles, batch=50):
    """Title -> raw wikitext, 50 pages per request, following redirects and continuations."""
    out = {}
    for i in tqdm(range(0, len(titles), batch), desc="wikipedia"):
        params = dict(action="query", prop="revisions", rvprop="content", rvslots="main", format="json",
                      formatversion=2, redirects=1, titles="|".join(titles[i:i + batch]))
        while True:
            js = http_json(WIKI_API, params=params)
            q = js.get("query", {})
            alias = {n["to"]: n["from"] for n in q.get("normalized", [])}
            for rd in q.get("redirects", []):
                alias[rd["to"]] = alias.get(rd["from"], rd["from"])
            for page in q.get("pages", []):
                if page.get("revisions"):
                    out[alias.get(page["title"], page["title"])] = page["revisions"][0]["slots"]["main"]["content"]
            if "continue" not in js:
                break
            params.update(js["continue"])
        time.sleep(0.2)
    return out

def _remove(code, node):
    try:
        code.remove(node)
    except ValueError:
        pass

def extract_plot(wikitext):
    if not wikitext:
        return None
    sections = mwparserfromhell.parse(wikitext).get_sections(levels=[2], matches=PLOT_HEADINGS)
    if not sections:
        return None
    sec = sections[0]
    for t in sec.filter_tags():
        if str(t.tag).strip().lower() in ("ref", "gallery"):
            _remove(sec, t)
    for link in sec.filter_wikilinks():
        if str(link.title).strip().lower().startswith(("file:", "image:")):
            _remove(sec, link)
    for h in sec.filter_headings():
        _remove(sec, h)
    text = sec.strip_code(normalize=True, collapse=True)
    text = re.sub(r"[ \t]+", " ", re.sub(r"\n{2,}", "\n\n", text)).strip()
    return text if len(text) >= CFG["min_plot_chars"] else None

def chunk_text(text, n_words):
    """Sentence-aligned chunks of about n_words, overlapping by one sentence."""
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text.replace("\n", " ")) if s.strip()]
    chunks, cur = [], []
    for s in sents:
        cur.append(s)
        if sum(len(x.split()) for x in cur) >= n_words:
            chunks.append(" ".join(cur))
            cur = cur[-1:]
    if len(cur) > 1 or (cur and not chunks):
        chunks.append(" ".join(cur))
    return chunks

def clean_title(t):
    """'Matrix, The (1999)' -> 'The Matrix (1999)'."""
    m = re.match(r"^(.*?), (The|A|An)( \(.*)$", t)
    return f"{m.group(2)} {m.group(1)}{m.group(3)}" if m else t

In [ ]:
def download_movielens():
    ml = WORK / "ml-25m"
    if not (ml / "ratings.csv").exists():
        zpath = WORK / "ml-25m.zip"
        with requests.get("https://files.grouplens.org/datasets/movielens/ml-25m.zip", stream=True, timeout=120) as r:
            r.raise_for_status()
            with zpath.open("wb") as f:
                for block in r.iter_content(1 << 20):
                    f.write(block)
        zipfile.ZipFile(zpath).extractall(WORK)
        zpath.unlink()
    return ml

def build_data(out):
    ml = download_movielens()
    ratings = pd.read_csv(ml / "ratings.csv", dtype={"userId": "int32", "movieId": "int32",
                                                     "rating": "float32", "timestamp": "int64"})
    movies = pd.read_csv(ml / "movies.csv")
    links = pd.read_csv(ml / "links.csv", dtype={"imdbId": str})

    movies["title_clean"] = movies.title.map(clean_title)
    movies[["movieId", "title_clean"]].to_parquet(out / "all_titles.parquet")  # for the out-of-context title check

    counts = ratings.movieId.value_counts()
    m = movies[movies.movieId.isin(counts[counts >= CFG["min_item_train_ratings"]].index)]
    m = m.merge(links[["movieId", "imdbId"]], on="movieId")
    m["imdb"] = "tt" + m.imdbId.str.zfill(7)
    # cache the slow fetches so a later failure doesn't repeat them
    wd_cache, wt_cache = WORK / "wikidata_titles.json", WORK / "wikitext.pkl"
    if not wd_cache.exists():
        wd_cache.write_text(json.dumps(wikidata_enwiki_titles(m.imdb.tolist())))
    m["wiki_title"] = m.imdb.map(json.loads(wd_cache.read_text()))
    if not wt_cache.exists():
        pd.to_pickle(fetch_wikitext(m.wiki_title.dropna().unique().tolist()), wt_cache)
    texts = pd.read_pickle(wt_cache)
    m["plot"] = [extract_plot(texts.get(t)) if isinstance(t, str) else None for t in m.wiki_title]
    print(f"{len(m)} movies with enough ratings | wikipedia article: {m.wiki_title.notna().mean():.1%} "
          f"| usable plot section: {m['plot'].notna().mean():.1%}")

    m = m[m["plot"].notna()].reset_index(drop=True)
    m[["movieId", "title", "title_clean", "genres", "imdb", "wiki_title", "plot"]].to_parquet(out / "movies.parquet")
    chunks = [(mid, j, c) for mid, plot in zip(m.movieId, m["plot"])
              for j, c in enumerate(chunk_text(plot, CFG["chunk_words"]))]
    pd.DataFrame(chunks, columns=["movieId", "chunk_no", "text"]).to_parquet(out / "chunks.parquet")
    ratings[ratings.movieId.isin(m.movieId)].to_parquet(out / "ratings.parquet")
    genome = pd.read_csv(ml / "genome-scores.csv", dtype={"movieId": "int32", "tagId": "int16", "relevance": "float32"})
    genome[genome.movieId.isin(m.movieId)].to_parquet(out / "genome.parquet")
    print(f"saved {len(m)} movies, {len(chunks)} chunks to {out}")

if not (DATA / "movies.parquet").exists():
    build_data(DATA)

## §2 Splits and catalog

For each user, sorted by time: **test** = last 5 ratings, **val** = the 5 before those, **train** = the rest. A rating ≥ 4 is a positive. The catalog is every movie with a plot and at least `min_item_train_ratings` train ratings, and every method uses this same catalog.

Protocols: tuning fits on **train** and scores on **val**. Final results refit on **train + val** and score on **test**. Items the user already rated in the fit window are never recommended.

*Known limitation:* a per-user split leaves other users' later ratings in training. This is standard in the literature and is kept for comparability.

In [ ]:
ratings = pd.read_parquet(DATA / "ratings.parquet")
movies = pd.read_parquet(DATA / "movies.parquet")
ratings = ratings[ratings.movieId.isin(movies.movieId)]
ratings = ratings.sort_values(["userId", "timestamp", "movieId"], kind="stable").reset_index(drop=True)
ratings = ratings[ratings.groupby("userId").movieId.transform("size") >= CFG["min_user_ratings"]]

H = CFG["holdout"]
from_end = ratings.groupby("userId").cumcount(ascending=False).values
ratings["split"] = pd.Categorical(np.where(from_end < H, "test", np.where(from_end < 2 * H, "val", "train")))

train_counts = ratings.loc[ratings.split == "train", "movieId"].value_counts()
catalog = np.sort(train_counts[train_counts >= CFG["min_item_train_ratings"]].index.values)
ratings = ratings[ratings.movieId.isin(catalog)].reset_index(drop=True)
users = np.unique(ratings.userId.values)
ratings["uid"] = np.searchsorted(users, ratings.userId.values).astype(np.int32)
ratings["iid"] = np.searchsorted(catalog, ratings.movieId.values).astype(np.int32)
n_users, n_items = len(users), len(catalog)

movies = movies.set_index("movieId").loc[catalog].reset_index()
item_title = movies.title_clean.tolist()
dupes = pd.Series(item_title).duplicated(keep=False).values
item_title = [f"{t} #{mid}" if d else t for t, d, mid in zip(item_title, dupes, movies.movieId)]
title2iid = {t: i for i, t in enumerate(item_title)}

def to_csr(df, positives_only):
    d = df[df.rating >= CFG["pos_threshold"]] if positives_only else df
    return sp.csr_matrix((np.ones(len(d), np.float32), (d.uid.values, d.iid.values)), shape=(n_users, n_items))

split = ratings.split.values
PROTO = {}
for name, fit_mask, target_mask in [("val", split == "train", split == "val"),
                                    ("test", split != "test", split == "test")]:
    fit, target = ratings[fit_mask], ratings[target_mask]
    fit_pos, target_pos = to_csr(fit, True), to_csr(target, True)
    eval_users = np.flatnonzero((np.diff(target_pos.indptr) > 0) & (np.diff(fit_pos.indptr) > 0))
    PROTO[name] = dict(fit=fit, fit_pos=fit_pos, seen=to_csr(fit, False), target_pos=target_pos,
                       eval_users=eval_users, pop=np.asarray(fit_pos.sum(0)).ravel())
    print(f"{name}: fit {len(fit):,} ratings | target {len(target):,} | eval users {len(eval_users):,}")

print(f"{n_users:,} users | {n_items:,} catalog items | {len(ratings):,} ratings")

## §3 Content embeddings and the Tag Genome check

Each ~200-word plot chunk is embedded with bge-m3 (it reads up to 8k tokens, so nothing is cut off). An item's vector is the mean of its chunk vectors. With about 40k chunks, a brute-force numpy dot product is fast enough, so FAISS isn't needed.

**Sanity check against human judgement:** the MovieLens Tag Genome scores each movie on 1,128 tags. If the embeddings capture content, movies that are close in embedding space should also be close in genome space.

In [ ]:
chunks = pd.read_parquet(DATA / "chunks.parquet")
chunks = chunks[chunks.movieId.isin(catalog)].reset_index(drop=True)
chunk_iid = np.searchsorted(catalog, chunks.movieId.values)
chunk_text_ = chunks.text.tolist()

emb_path = find_input("chunk_emb.npy") or WORK / "chunk_emb.npy"
if emb_path.exists():
    E = np.load(emb_path)
else:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer(CFG["embed_model"], device=str(DEV)).half()
    enc.max_seq_length = 512
    E = enc.encode(chunk_text_, batch_size=64, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
    np.save(emb_path, E)
    del enc
    free_gpu()

pool = sp.csr_matrix((np.ones(len(chunk_iid), np.float32), (chunk_iid, np.arange(len(chunk_iid)))),
                     shape=(n_items, len(chunk_iid)))
item_vec = pool @ E
item_vec /= np.linalg.norm(item_vec, axis=1, keepdims=True)
item_chunks = np.split(np.arange(len(chunk_iid)), np.cumsum(np.bincount(chunk_iid, minlength=n_items))[:-1])
print(E.shape, item_vec.shape, "chunks per item:", np.bincount(chunk_iid).mean().round(2))

In [ ]:
from scipy.stats import spearmanr

g = pd.read_parquet(DATA / "genome.parquet")
G = g[g.movieId.isin(catalog)].pivot(index="movieId", columns="tagId", values="relevance")
gi = np.searchsorted(catalog, G.index.values)
Gv = G.values - G.values.mean(0)                     # centre tags, or every pair looks similar
Gv /= np.linalg.norm(Gv, axis=1, keepdims=True)
Ev = item_vec[gi]

rng = np.random.default_rng(SEED)
a, b = rng.integers(0, len(gi), (2, 50_000))
keep = a != b
rho = spearmanr((Ev[a] * Ev[b]).sum(1)[keep], (Gv[a] * Gv[b]).sum(1)[keep]).correlation

def knn(X, k=10, bs=2048):
    out = np.empty((len(X), k), np.int64)
    for s in range(0, len(X), bs):
        S = X[s:s + bs] @ X.T
        S[np.arange(S.shape[0]), np.arange(s, s + S.shape[0])] = -np.inf
        out[s:s + bs] = np.argpartition(-S, k, axis=1)[:, :k]
    return out

ke, kg = knn(Ev), knn(Gv)
overlap = np.mean([len(set(x) & set(y)) / 10 for x, y in zip(ke, kg)])
print(f"{len(gi):,} catalog movies have genome scores")
print(f"Spearman(embedding sim, genome sim) over random pairs: {rho:.3f}")
print(f"10-NN overlap embedding vs genome: {overlap:.3f}  (random would be {10/(len(gi)-1):.4f})")

## §4 CF baselines and the evaluator

Popularity, item-kNN (cosine, top-k neighbours), EASE (closed form) and implicit ALS. Each is tuned on **val** and then refit on train+val for **test**. All items in the catalog are ranked; nothing is sampled.

Metrics:
- **Recall@10:** hits ÷ min(10, number of relevant items).
- **NDCG@10:** binary relevance.
- **Coverage:** share of the catalog that is recommended to anyone.
- **ARP:** mean training popularity of the recommended items.
- **Tail share:** share of recommendations outside the 20% most popular items.

CIs come from bootstrapping over users.

In [ ]:
def gram(X, bs=20_000):
    G = torch.zeros(X.shape[1], X.shape[1], device=DEV)
    for s in range(0, X.shape[0], bs):
        Xb = torch.from_numpy(X[s:s + bs].toarray()).to(DEV)
        G.addmm_(Xb.T, Xb)
    return G

def dense_rows(X, rows):
    return torch.from_numpy(X[rows].toarray()).to(DEV)

class Popularity:
    def fit(self, X, G=None):
        self.s = torch.from_numpy(np.asarray(X.sum(0)).ravel()).to(DEV)
        return self
    def score(self, X, rows):
        return self.s.expand(len(rows), -1).clone()

class ItemKNN:
    def __init__(self, k):
        self.k = k
    def fit(self, X, G):
        n = torch.sqrt(torch.diag(G))
        S = G / (n[:, None] * n[None, :] + 1e-8)
        S.fill_diagonal_(0)
        vals, idx = S.topk(self.k, dim=0)               # keep each item's k nearest neighbours
        self.S = torch.zeros_like(S).scatter_(0, idx, vals)
        return self
    def score(self, X, rows):
        return dense_rows(X, rows) @ self.S

class EASE:
    def __init__(self, lam):
        self.lam = lam
    def fit(self, X, G):
        P = torch.linalg.inv(G + self.lam * torch.eye(G.shape[0], device=DEV))
        self.B = -P / torch.diag(P)[None, :]
        self.B.fill_diagonal_(0)
        return self
    def score(self, X, rows):
        return dense_rows(X, rows) @ self.B

class ALS:
    def __init__(self, factors, alpha, reg=0.05, iters=15):
        self.kw = dict(factors=factors, alpha=alpha, regularization=reg, iterations=iters)
    def fit(self, X, G=None):
        from implicit.cpu.als import AlternatingLeastSquares
        from threadpoolctl import threadpool_limits
        m = AlternatingLeastSquares(random_state=SEED, **self.kw)
        with threadpool_limits(1, "blas"):  # implicit runs its own threads; nested BLAS threads slow it down
            m.fit(X.astype(np.float32), show_progress=False)
        self.U = torch.from_numpy(np.asarray(m.user_factors)).to(DEV)
        self.V = torch.from_numpy(np.asarray(m.item_factors)).to(DEV)
        return self
    def score(self, X, rows):
        return self.U[torch.as_tensor(rows, device=DEV)] @ self.V.T

def recommend(model, proto, rows, k, bs=4096):
    fit_pos, seen = proto["fit_pos"], proto["seen"]
    out = np.empty((len(rows), k), np.int64)
    for s in range(0, len(rows), bs):
        rb = rows[s:s + bs]
        S = model.score(fit_pos, rb).float()
        m = seen[rb].tocoo()
        S[torch.as_tensor(m.row, device=DEV), torch.as_tensor(m.col, device=DEV)] = -torch.inf
        out[s:s + bs] = S.topk(k, dim=1).indices.cpu().numpy()
    return out

def per_user_metrics(recs, rows, target_pos):
    k = recs.shape[1]
    T = target_pos[rows]
    hits = np.asarray(T[np.arange(len(rows))[:, None], recs].todense()) > 0
    n_rel = np.diff(T.indptr)
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    idcg = np.cumsum(disc)[np.minimum(n_rel, k) - 1]
    return dict(recall=hits.sum(1) / np.minimum(n_rel, k), ndcg=(hits * disc).sum(1) / idcg)

def list_metrics(recs, pop):
    tail = pop < np.quantile(pop, 0.8)
    return dict(coverage=len(np.unique(recs)) / n_items, arp=pop[recs].mean(), tail_share=tail[recs].mean())

def boot_ci(x, n_boot=CFG["n_boot"], seed=SEED):
    rng = np.random.default_rng(seed)
    means = np.array([x[rng.integers(0, len(x), len(x))].mean() for _ in range(n_boot)])
    return x.mean(), *np.quantile(means, [0.025, 0.975])

In [ ]:
GRID = {"Popularity": [{}],
        "ItemKNN": [dict(k=k) for k in (50, 200)],
        "EASE": [dict(lam=l) for l in (100, 300, 1000, 3000)],
        "ALS": [dict(factors=f, alpha=a) for f in (64, 128) for a in (10, 40)]}
CLASSES = {"Popularity": Popularity, "ItemKNN": ItemKNN, "EASE": EASE, "ALS": ALS}

val = PROTO["val"]
G_val = gram(val["fit_pos"])
tuning = []
for name, configs in GRID.items():
    for cfg in configs:
        t0 = time.time()
        model = CLASSES[name](**cfg).fit(val["fit_pos"], G_val)
        recs = recommend(model, val, val["eval_users"], CFG["topk"])
        ndcg = per_user_metrics(recs, val["eval_users"], val["target_pos"])["ndcg"].mean()
        tuning.append(dict(model=name, cfg=json.dumps(cfg), val_ndcg10=ndcg, secs=time.time() - t0))
        print(f"{name:10s} {cfg}  val NDCG@10 = {ndcg:.4f}")
        del model
        free_gpu()
tuning = pd.DataFrame(tuning)
tuning.to_csv(RES / "cf_tuning.csv", index=False)
best_cfg = {m: json.loads(d.loc[d.val_ndcg10.idxmax(), "cfg"]) for m, d in tuning.groupby("model")}
CAND_MODEL = tuning.loc[tuning.val_ndcg10.idxmax(), "model"]
print("best configs:", best_cfg, "| candidate generator:", CAND_MODEL)

Refit each baseline with its best config on train+val and score on test. The best model by val NDCG becomes the **candidate generator**: its top-50 goes to Speculative RAG. Its Recall@50 is the ceiling, since a reranker can only reorder what CF retrieved.

The candidate generator is also fit on the val protocol, for users outside the test sample. §10 uses those candidates to make teacher examples.

In [ ]:
test = PROTO["test"]
rng = np.random.default_rng(SEED)
SAMPLE = np.sort(rng.choice(test["eval_users"], CFG["n_eval_users"], replace=False))
np.save(RES / "sample_users.npy", SAMPLE)

G_test = gram(test["fit_pos"])
cf_rows, cf_recs = [], {}
for name in GRID:
    model = CLASSES[name](**best_cfg[name]).fit(test["fit_pos"], G_test)
    recs = recommend(model, test, test["eval_users"], CFG["topk"])
    m = per_user_metrics(recs, test["eval_users"], test["target_pos"])
    cf_recs[name] = dict(zip(test["eval_users"], recs))
    cf_rows.append(dict(model=name, users="all", n=len(recs),
                        **{f"{k}@10": boot_ci(v) for k, v in m.items()}, **list_metrics(recs, test["pop"])))
    if name == CAND_MODEL:
        cands_test = recommend(model, test, SAMPLE, CFG["n_cand"])
        m50 = per_user_metrics(cands_test, SAMPLE, test["target_pos"])
        print(f"candidate Recall@50 on the sample (reranking ceiling): {m50['recall'].mean():.4f}")
    del model
    free_gpu()

# teacher pool for §10: val-protocol candidates for users outside the test sample
TEACH_POOL = np.setdiff1d(val["eval_users"], SAMPLE)
TEACH_POOL = np.sort(np.random.default_rng(SEED + 1).choice(TEACH_POOL, min(len(TEACH_POOL), CFG["n_teacher"] * 3), replace=False))
cand_val_model = CLASSES[CAND_MODEL](**best_cfg[CAND_MODEL]).fit(val["fit_pos"], G_val)
cands_teach = recommend(cand_val_model, val, TEACH_POOL, CFG["n_cand"])
del cand_val_model, G_val, G_test
free_gpu()

fmt = lambda t: f"{t[0]:.4f} [{t[1]:.4f}, {t[2]:.4f}]" if isinstance(t, tuple) else t
cf_table = pd.DataFrame(cf_rows)
cf_table.to_pickle(RES / "cf_test_all_users.pkl")
display(cf_table.map(fmt))

## §5 User queries

For each sampled user (test protocol, so the history is train + val):
- **Q:** the last 15 movies they rated highly and the last 5 they rated ≤ 2. Dislikes appear as dislikes, never as weak likes.
- **Retrieval profile (Rocchio):** mean of liked item vectors − β × mean of disliked item vectors.

In [ ]:
def build_contexts(user_rows, proto, cand_lists, seed=SEED):
    hist = proto["fit"]
    hist = hist[hist.uid.isin(user_rows)].groupby("uid")
    ctxs = {}
    for u, cands in zip(user_rows, cand_lists):
        h = hist.get_group(u)
        liked = h.iid.values[h.rating.values >= CFG["pos_threshold"]][::-1][:CFG["n_liked_in_query"]]
        disliked = h.iid.values[h.rating.values <= CFG["dislike_threshold"]][::-1][:CFG["n_disliked_in_query"]]
        prof = item_vec[liked].mean(0)
        if len(disliked):
            prof = prof - CFG["rocchio_beta"] * item_vec[disliked].mean(0)
        prof /= np.linalg.norm(prof)
        rng = np.random.default_rng([seed, int(u)])
        ctxs[int(u)] = dict(u=int(u), cands=[int(c) for c in cands], liked=liked.tolist(),
                            disliked=disliked.tolist(), profile=prof,
                            order_full=rng.permutation(cands).tolist(), rng_seed=[seed, int(u)])
    return ctxs

CTX = build_contexts(SAMPLE, test, cands_test)
print(len(CTX), "contexts; example liked:", [item_title[i] for i in CTX[int(SAMPLE[0])]["liked"][:5]])

## §6 Evidence and draft subsets

The paper's recipe, applied to this task:
1. The evidence pool is the plot chunks of the 50 candidates and of the user's 5 most recent liked movies.
2. Retrieve the top `n_docs = 10` chunks by similarity to the profile (the paper retrieves 10).
3. K-means them into `k = 3` clusters.
4. Build `m = 5` unique subsets by sampling one chunk per cluster.

Each draft ranks **all** 50 candidates. Only the evidence differs between drafts, as in the paper. Keeping everything else identical also lets vLLM cache the shared start of a user's prompts.

Two arms differ only in how the candidates are listed:
- **Shuffled** (run 1): one random order per user, so the model can't copy the CF ranking.
- **CF order** (run 2): numbered in EASE's order, with a sentence saying where the order comes from.

If run 1 is attached, its contexts are reused, so its drafts line up with this run's users, evidence and subsets.

In [ ]:
from sklearn.cluster import KMeans

def sample_subsets(labels, k, m, rng, max_tries=500):
    clusters = [np.flatnonzero(labels == c) for c in range(k)]
    seen, out = set(), []
    for _ in range(max_tries):
        s = tuple(sorted(int(rng.choice(c)) for c in clusters))
        if s not in seen:
            seen.add(s)
            out.append(list(s))
            if len(out) == m:
                break
    return out

def add_evidence(ctx):
    rng = np.random.default_rng(ctx["rng_seed"])
    items = ctx["cands"] + ctx["liked"][:CFG["n_liked_evidence"]]
    pool_ids = np.concatenate([item_chunks[i] for i in items])
    sims = E[pool_ids] @ ctx["profile"]
    top = pool_ids[np.argsort(-sims)[:CFG["n_docs"]]]
    labels = KMeans(CFG["k_clusters"], n_init=5, random_state=SEED).fit_predict(E[top])
    ctx["evidence"] = top.tolist()
    ctx["subsets"] = [[int(top[j]) for j in s] for s in sample_subsets(labels, CFG["k_clusters"], CFG["m_drafts"], rng)]
    return ctx

for ctx in tqdm(CTX.values(), desc="evidence"):
    add_evidence(ctx)
if PREV and (PREV / "contexts_test.pkl").exists():
    prev = pd.read_pickle(PREV / "contexts_test.pkl")
    assert set(prev) == set(CTX), "run 1 sampled different users; detach its outputs or rerun both arms"
    same = np.mean([prev[u]["cands"] == c["cands"] and prev[u]["subsets"] == c["subsets"] for u, c in CTX.items()])
    print(f"run 1 contexts: {same:.1%} identical to a fresh build; using run 1's so its outputs line up")
    CTX = prev
pd.to_pickle(CTX, RES / "contexts_test.pkl")

def cf_order_view(ctxs):
    """The CF-order arm: the same contexts, with candidates listed in the CF model's order and numbered."""
    return {u: {**c, "order_full": list(c["cands"]), "cf_ranked": True} for u, c in ctxs.items()}

CTX_CF = cf_order_view(CTX)
print("drafts per user:", pd.Series([len(c["subsets"]) for c in CTX.values()]).value_counts().to_dict())

### Prompts

The same builders serve every LLM system, so the ablation varies only the evidence and the model.
- **Drafter / single-call systems:** user history + candidates + task + evidence. The parts shared by all of a user's drafts come first, so vLLM's prefix cache computes them once. In the CF-order arm, the candidates are numbered and the task adds one sentence saying where the order comes from. It informs the model about the order; it doesn't tell it to keep it.
- **Verifier Q:** user history + task only. The paper's verifier sees the question without retrieved material; here both the plot evidence and the CF candidate list are retrieval output, so neither is shown. The verifier judges the picks with its own knowledge of the films.

In [ ]:
SYSTEM = ("You are a movie recommender. You rerank one user's candidate movies using their rating history, "
          "the plot evidence provided, and what you know about the films.")
K = CFG["topk"]
DRAFT_TASK = (f"Choose the {K} candidates this user is most likely to enjoy next, best first. First write a rationale "
              f"of at most 120 words that connects the user's taste to your picks and cites evidence numbers like [2] "
              f"where they apply; then give the ranking, using candidate titles exactly as listed and each title only once.")
VERIFY_TASK = (f"Recommend the {K} movies this user is most likely to enjoy next, best first. "
               f"Give a short rationale, then the ranking, each title only once.")
REFLECT = "Do you think the explanation supports the answers? (Yes or No)"
CF_NOTE = ("The candidates are numbered in the order a collaborative-filtering model, trained on many users' ratings, "
           "ranks them for this user, most likely first. Treat that order as one signal alongside the history and the evidence.")

def history_block(ctx):
    liked = "; ".join(item_title[i] for i in ctx["liked"]) or "none recorded"
    disliked = "; ".join(item_title[i] for i in ctx["disliked"]) or "none recorded"
    return f"## User history\nRated highly (most recent first): {liked}\nDisliked: {disliked}"

def evidence_block(ctx, chunk_ids):
    cand = set(ctx["cands"])
    lines = [f"[{n}] {item_title[chunk_iid[c]]} ({'candidate' if chunk_iid[c] in cand else 'the user liked this'}): "
             f"{chunk_text_[c]}" for n, c in enumerate(chunk_ids, 1)]
    return "## Plot evidence\n" + "\n\n".join(lines)

def candidate_block(order, ranked=False):
    if ranked:
        return ("## Candidates (collaborative-filtering order, most likely first)\n"
                + "\n".join(f"{r}. {item_title[i]}" for r, i in enumerate(order, 1)))
    return "## Candidates\n" + "\n".join(f"- {item_title[i]}" for i in order)

def drafter_messages(ctx, chunk_ids):
    ranked = ctx.get("cf_ranked", False)
    task = DRAFT_TASK + (" " + CF_NOTE if ranked else "")
    body = "\n\n".join([history_block(ctx), candidate_block(ctx["order_full"], ranked), task, evidence_block(ctx, chunk_ids)])
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": body}]

def question_messages(ctx, extra=""):
    body = "\n\n".join([history_block(ctx), VERIFY_TASK]) + extra
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": body}]

def draft_schema(order):
    return {"type": "object", "additionalProperties": False, "required": ["rationale", "ranking"],
            "properties": {"rationale": {"type": "string"},
                           "ranking": {"type": "array", "minItems": K, "maxItems": K,
                                       "items": {"type": "string", "enum": [item_title[i] for i in order]}}}}

def parse_draft(text):
    """-> (rationale, ranked iids without duplicates, stats). Titles outside the enum can't occur under the schema."""
    try:
        obj = json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None, [], dict(valid=False, dups=0)
    ranking, dups = [], 0
    for t in obj.get("ranking", []):
        i = title2iid.get(t)
        if i is None:
            continue
        if i in ranking:
            dups += 1
        else:
            ranking.append(i)
    return obj.get("rationale", ""), ranking, dict(valid=True, dups=dups)

def backfill(ranking, cands, k=K):
    return (list(ranking) + [c for c in cands if c not in ranking])[:k]

ex = CTX_CF[int(SAMPLE[0])]
print(drafter_messages(ex, ex["subsets"][0])[1]["content"][:2500])

## §7 Drafter

Requests go to the vLLM servers in parallel, and results are appended to a JSONL file as they arrive. A crashed or interrupted run resumes where it stopped, and failed requests are retried on the next run.

**Drafting score**, as in the paper: ρ_Draft = P(β | Q, δ) + P(α | Q, δ, β), where β is the rationale and α is the ranking. Each term is the per-token geometric mean of the generated tokens' probabilities, i.e. length-normalised, because rationales vary in length. The split between β and α is at the `"ranking"` key of the JSON.

In [ ]:
async def post_json(client, url, payload, retries=4):
    for attempt in range(retries):
        try:
            r = await client.post(url, json=payload)
            r.raise_for_status()
            return r.json()
        except httpx.HTTPError:
            if attempt == retries - 1:
                raise
            await asyncio.sleep(2 ** attempt)

async def run_requests(items, fn, path, concurrency, desc):
    """items: [(key, payload)]. Each result is appended to `path` when it finishes; errors are retried next run."""
    done = {}
    if path.exists():
        for line in path.open():
            r = json.loads(line)
            if "error" not in r and r.get("text", "") is not None:  # a null text (run 1's empty Gemini replies) is redone
                done[r["key"]] = r
    todo = [(k, p) for k, p in items if k not in done]
    sem = asyncio.Semaphore(concurrency)
    t0 = time.perf_counter()
    with path.open("a") as f, tqdm(total=len(todo), desc=desc) as bar:
        async def one(k, p):
            async with sem:
                try:
                    r = await fn(p)
                except Exception as e:
                    r = {"error": repr(e)}
            r["key"] = k
            f.write(json.dumps(r) + "\n")
            f.flush()
            if "error" not in r:
                done[k] = r
            bar.update()
        await asyncio.gather(*(one(k, p) for k, p in todo))
    n_err = len(todo) - sum(k in done for k, _ in todo)
    if todo:
        print(f"{desc}: {len(todo)} requests in {time.perf_counter() - t0:.0f}s, {n_err} errors")
    return done, (time.perf_counter() - t0 if todo else None)

def draft_log_score(tokens, text):
    """log ρ_Draft from the generated tokens' logprobs, split at the "ranking" key."""
    pos = text.find('"ranking"')
    if pos < 0 or not tokens:
        return None
    offs = np.cumsum([0] + [len(t["token"]) for t in tokens])[:-1]
    lps = np.array([t["logprob"] for t in tokens])
    rat, ans = lps[offs < pos], lps[offs >= pos]
    if not len(rat) or not len(ans):
        return None
    return float(np.log(np.exp(rat.mean()) + np.exp(ans.mean())))

def make_draft_fn(client, urls, model):
    async def fn(p):
        url = urls[p["u"] % len(urls)]  # same user -> same server, so its shared prompt prefix is cached
        t0 = time.perf_counter()
        r = await post_json(client, f"{url}/v1/chat/completions", {
            "model": model, "messages": p["messages"], "temperature": 0, "max_tokens": 700, "seed": SEED,
            "logprobs": True, "response_format": {"type": "json_schema", "json_schema": {"name": "draft", "schema": p["schema"]}}})
        ch = r["choices"][0]
        text = ch["message"]["content"]
        return dict(text=text, finish=ch["finish_reason"], latency=time.perf_counter() - t0,
                    in_tok=r["usage"]["prompt_tokens"], out_tok=r["usage"]["completion_tokens"],
                    log_rho_draft=draft_log_score((ch.get("logprobs") or {}).get("content") or [], text))
    return fn

def draft_items(ctxs, single=False):
    """Spec RAG drafts (one per evidence subset), or a single draft over all evidence (`single=True`)."""
    items = []
    for c in ctxs.values():
        if single:
            items.append((f"{c['u']}:all", dict(u=c["u"], messages=drafter_messages(c, c["evidence"]),
                                                 schema=draft_schema(c["order_full"]))))
        else:
            for j, sub in enumerate(c["subsets"]):
                items.append((f"{c['u']}:{j}", dict(u=c["u"], messages=drafter_messages(c, sub), schema=draft_schema(c["order_full"]))))
    return items

## §8 Verifier

The verifier is **Qwen2.5-14B-AWQ**. It isn't fine-tuned and it sees neither the evidence nor the candidate list, matching the paper's generalist verifier, which is larger than the drafter.
- **ρ_SC** = P(α, β | Q): teacher-forced log-probability of the draft given only the question, as a per-token mean. The paper uses the raw product; normalising by length keeps long rationales from being penalised.
- **ρ_SR** = P("Yes" | Q, α, β, R), with R = *"Do you think the explanation supports the answers? (Yes or No)"*, the paper's own wording.
- **Score:** log ρ = log ρ_Draft + log ρ_SC + log ρ_SR, and the highest-scoring draft wins.

Prompts are tokenised locally and sent as token ids, so the scored span is exactly the draft's tokens.

In [ ]:
vtok = AutoTokenizer.from_pretrained(CFG["verifier_model"])

def canonical_answer(rationale, ranking):
    return json.dumps({"rationale": rationale, "ranking": [item_title[i] for i in ranking]}, ensure_ascii=False)

def sc_payload(ctx, rationale, ranking):
    prefix = vtok.apply_chat_template(question_messages(ctx), add_generation_prompt=True, tokenize=False)
    enc = vtok(prefix + canonical_answer(rationale, ranking), add_special_tokens=False, return_offsets_mapping=True)
    span = [i for i, (s, _) in enumerate(enc["offset_mapping"]) if s >= len(prefix)]
    return dict(kind="sc", ids=enc["input_ids"], span=span)

def sr_payload(ctx, rationale, ranking):
    extra = (f"\n\n## Proposed answer\nRanking: {'; '.join(item_title[i] for i in ranking)}\n"
             f"Explanation: {rationale}\n\n{REFLECT}")
    prompt = vtok.apply_chat_template(question_messages(ctx, extra), add_generation_prompt=True, tokenize=False)
    return dict(kind="sr", ids=vtok(prompt, add_special_tokens=False)["input_ids"])

def make_verify_fn(client, urls):
    async def fn(p):
        url = urls[p["u"] % len(urls)]
        t0 = time.perf_counter()
        if p["kind"] == "sc":
            r = await post_json(client, f"{url}/v1/completions", {
                "model": "verifier", "prompt": p["ids"], "max_tokens": 1, "temperature": 0, "prompt_logprobs": 0})
            pl = r["choices"][0]["prompt_logprobs"]
            lps = [pl[i][str(p["ids"][i])]["logprob"] for i in p["span"]]
            score = float(np.mean(lps))
        else:
            r = await post_json(client, f"{url}/v1/completions", {
                "model": "verifier", "prompt": p["ids"], "max_tokens": 1, "temperature": 0, "logprobs": 20})
            top = r["choices"][0]["logprobs"]["top_logprobs"][0]
            p_yes = sum(np.exp(lp) for t, lp in top.items() if t.strip().lower() == "yes")
            score = float(np.log(max(p_yes, 1e-8)))
        return dict(score=score, latency=time.perf_counter() - t0)
    return fn

def verify_items(ctxs, drafts):
    items = []
    for c in ctxs.values():
        for j in range(len(c["subsets"])):
            d = drafts.get(f"{c['u']}:{j}")
            if not d:
                continue
            rationale, ranking, st = parse_draft(d["text"])
            if not st["valid"] or not ranking:
                continue
            items.append((f"{c['u']}:{j}:sc", dict(sc_payload(c, rationale, ranking), u=c["u"])))
            items.append((f"{c['u']}:{j}:sr", dict(sr_payload(c, rationale, ranking), u=c["u"])))
    return items

### Selecting a draft

- **Spec RAG (argmax):** the paper's rule; the best draft's ranking is the answer.
- **Spec RAG (Borda):** every draft votes, weighted by softmax(log ρ). This variant is an ablation, not part of the paper.
- **Spec RAG + RRF:** reciprocal-rank fusion of the Borda list with the CF order. It has no parameters, so nothing is tuned on test.

Invalid or empty drafts drop out. If no draft for a user is valid, that user falls back to the CF order, and the fallback is counted.

In [ ]:
def combine(ctxs, drafts, verdicts):
    out = {}
    for c in ctxs.values():
        u, cands, cand_rank = c["u"], c["cands"], {i: r for r, i in enumerate(c["cands"])}
        scored = []
        for j in range(len(c["subsets"])):
            d = drafts.get(f"{u}:{j}")
            sc, sr = verdicts.get(f"{u}:{j}:sc"), verdicts.get(f"{u}:{j}:sr")
            if not d or not sc or not sr or d["log_rho_draft"] is None:
                continue
            rationale, ranking, _ = parse_draft(d["text"])
            if ranking:
                scored.append(dict(j=j, rationale=rationale, ranking=ranking, log_rho=d["log_rho_draft"] + sc["score"] + sr["score"],
                                   parts=(d["log_rho_draft"], sc["score"], sr["score"])))
        if not scored:
            out[u] = dict(argmax=cands[:K], borda=cands[:K], rrf=cands[:K], fallback=True, best=None, scored=[])
            continue
        best = max(scored, key=lambda s: s["log_rho"])
        w = np.exp(np.array([s["log_rho"] for s in scored]) - max(s["log_rho"] for s in scored))
        w /= w.sum()
        votes = {}
        for wi, s in zip(w, scored):
            for r, i in enumerate(s["ranking"]):
                votes[i] = votes.get(i, 0) + wi * (K - r)
        borda = sorted(votes, key=votes.get, reverse=True)
        rrf = {i: 1 / (60 + cand_rank[i]) for i in cands}
        for r, i in enumerate(borda):
            rrf[i] += 1 / (60 + r)
        out[u] = dict(argmax=backfill(best["ranking"], cands), borda=backfill(borda, cands),
                      rrf=sorted(rrf, key=rrf.get, reverse=True)[:K], fallback=False,
                      best=dict(j=best["j"], rationale=best["rationale"], evidence=c["subsets"][best["j"]]), scored=scored)
    return out

### Run: one stage at a time, both GPUs per stage

On T4s the work is limited by prompt processing, so each stage runs one server per GPU instead of leaving a GPU idle:
1. **Drafting** (two drafter servers): single drafts over all the evidence (the *CF + drafter* ablation), then the m Spec RAG drafts for each user.
2. **Verifying** (two verifier servers): ρ_SC and ρ_SR for every valid draft.

A stage whose results are already complete doesn't start its servers, so with run 1 attached the shuffled arm costs nothing and only the CF-order arm runs. Wall time is recorded for throughput. Batch-size-1 latency is measured separately in the next cell, with one drafter and one verifier running side by side, as a deployed system would.

In [ ]:
def n_pending(items, path):
    done = set()
    if path.exists():
        done = {r["key"] for r in map(json.loads, path.open()) if "error" not in r}
    return sum(k not in done for k, _ in items)

async def run_specrag(ctxs, tag, drafter_model_name="drafter", drafter_args=DRAFTER_ARGS):
    walls = {}
    single_items, draft_items_ = draft_items(ctxs, single=True), draft_items(ctxs)
    paths = dict(single=RES / f"single_{tag}.jsonl", drafts=RES / f"drafts_{tag}.jsonl", verify=RES / f"verify_{tag}.jsonl")
    async with httpx.AsyncClient(timeout=1800, limits=httpx.Limits(max_connections=64)) as client:
        pool = []
        if n_pending(single_items, paths["single"]) or n_pending(draft_items_, paths["drafts"]):
            pool = start_pool(CFG["drafter_model"], "drafter", drafter_args, GPUS, 8000)
        try:
            dfn = make_draft_fn(client, [srv.url for srv in pool], drafter_model_name)
            single, walls["single"] = await run_requests(single_items, dfn, paths["single"], 16 * len(GPUS), f"single draft [{tag}]")
            drafts, walls["drafts"] = await run_requests(draft_items_, dfn, paths["drafts"], 16 * len(GPUS), f"spec drafts [{tag}]")
        finally:
            stop_pool(pool)

        v_items, pool = verify_items(ctxs, drafts), []
        if n_pending(v_items, paths["verify"]):
            pool = start_pool(CFG["verifier_model"], "verifier", VERIFIER_ARGS, GPUS, 8100)
        try:
            vfn = make_verify_fn(client, [srv.url for srv in pool])
            verdicts, walls["verify"] = await run_requests(v_items, vfn, paths["verify"], 8 * len(GPUS), f"verifier [{tag}]")
        finally:
            stop_pool(pool)
    walls_path = RES / f"walltime_{tag}.json"
    # throughput is exact only for an uninterrupted pass; a resumed pass times just the remainder
    if walls_path.exists():
        walls = {k: v if v is not None else json.loads(walls_path.read_text()).get(k) for k, v in walls.items()}
    walls_path.write_text(json.dumps(walls))
    return single, drafts, verdicts, walls

# each vLLM server needs several GB of CPU RAM; drop the large frames nothing after §6 uses
for name in ("ratings", "g", "G", "Gv", "Ev", "ke", "kg", "chunks"):
    globals().pop(name, None)
gc.collect()
free_gpu()

single_base, drafts_base, verdicts_base, walls_base = await run_specrag(CTX, "base")
SPEC_BASE = combine(CTX, drafts_base, verdicts_base)
pd.to_pickle(SPEC_BASE, RES / "spec_base.pkl")
print("shuffled arm | fallback users:", sum(v["fallback"] for v in SPEC_BASE.values()), "| wall times (s):", walls_base)

single_cf, drafts_cf, verdicts_cf, walls_cf = await run_specrag(CTX_CF, "cf")
SPEC_CF = combine(CTX_CF, drafts_cf, verdicts_cf)
pd.to_pickle(SPEC_CF, RES / "spec_cf.pkl")
print("CF-order arm | fallback users:", sum(v["fallback"] for v in SPEC_CF.values()), "| wall times (s):", walls_cf)

In [ ]:
async def latency_probe(ctxs, users, drafter_model_name="drafter", drafter_args=DRAFTER_ARGS):
    """Batch-size-1 latency: one user at a time, drafts in parallel within the user, as in the paper.
    Drafter on GPU0 and verifier on GPU1 run side by side, as a deployed system would."""
    rows, dpool, vpool = [], start_pool(CFG["drafter_model"], "drafter", drafter_args, [0], 8000), []
    try:
        vpool = start_pool(CFG["verifier_model"], "verifier", VERIFIER_ARGS, [1], 8100)
        async with httpx.AsyncClient(timeout=1800) as client:
            dfn = make_draft_fn(client, [srv.url for srv in dpool], drafter_model_name)
            vfn = make_verify_fn(client, [srv.url for srv in vpool])
            for u in tqdm(users, desc="latency"):
                c = ctxs[int(u)]
                t0 = time.perf_counter()
                await dfn(draft_items({c["u"]: c}, single=True)[0][1])
                t_single = time.perf_counter() - t0

                t0 = time.perf_counter()
                items = draft_items({c["u"]: c})
                res = await asyncio.gather(*(dfn(p) for _, p in items))
                v_items = verify_items({c["u"]: c}, {k: r for (k, _), r in zip(items, res)})
                await asyncio.gather(*(vfn(p) for _, p in v_items))
                rows.append(dict(u=c["u"], single=t_single, specrag=time.perf_counter() - t0))
    finally:
        stop_pool(dpool + vpool)
    return pd.DataFrame(rows)

# The CF-order prompt is ~50 tokens longer (the numbering), so the shuffled arm's latency stands for both arms.
LAT_USERS = SAMPLE[:CFG["n_latency_users"]]
if (RES / "latency_base.csv").exists():
    lat_base = pd.read_csv(RES / "latency_base.csv")
else:
    lat_base = await latency_probe(CTX, LAT_USERS)
    lat_base.to_csv(RES / "latency_base.csv", index=False)
print(lat_base[["single", "specrag"]].describe(percentiles=[0.5, 0.95]).round(2))

## §9 Gemini single-call baseline

Gemini gets the same prompt as the single drafter: all the evidence chunks plus the candidates, with output restricted by schema to candidate titles. Token usage, including thinking tokens, is logged for cost. It runs once per arm.

In run 1, 45 of 500 calls returned no text and those users silently got the CF order. Now the finish reason and block reason are logged. An empty reply is retried, and the last attempt drops the title enum from the schema (marked `relaxed`; titles that don't match a candidate are ignored). A user who still gets nothing is an error that the next run retries, and the table reports how many users Gemini answered.

Gemini is **not** used as the verifier: current Flash models don't return logprobs, and the paper's scores need them.

In [ ]:
from google import genai
from google.genai import types, errors

gclient = genai.Client(api_key=GEMINI_API_KEY)

def gemini_schema(order, with_ranking=True):
    props = {"rationale": {"type": "STRING"}}
    if with_ranking:
        props["ranking"] = {"type": "ARRAY", "items": {"type": "STRING", "enum": [item_title[i] for i in order]}}
    return {"type": "OBJECT", "properties": props, "required": list(props), "propertyOrdering": list(props)}

def relaxed_schema(schema):
    """The same schema without the title enum, for a prompt Gemini keeps answering with nothing."""
    s = json.loads(json.dumps(schema))
    if "ranking" in s["properties"]:
        s["properties"]["ranking"]["items"].pop("enum", None)
    return s

async def gemini_call(p, retries=6, empty_retries=3):
    tried = []
    for e_try in range(empty_retries):
        relaxed = e_try == empty_retries - 1
        cfg = types.GenerateContentConfig(system_instruction=p["system"], temperature=0, response_mime_type="application/json",
                                          response_schema=relaxed_schema(p["schema"]) if relaxed else p["schema"])
        for attempt in range(retries):
            try:
                t0 = time.perf_counter()
                r = await gclient.aio.models.generate_content(model=CFG["gemini_model"], contents=p["user"], config=cfg)
                break
            except errors.APIError as e:
                if e.code not in (429, 500, 503) or attempt == retries - 1:
                    raise
                await asyncio.sleep(min(60, 2 ** attempt * 2))
        finish = str(r.candidates[0].finish_reason) if r.candidates else None
        block = str(r.prompt_feedback.block_reason) if r.prompt_feedback and r.prompt_feedback.block_reason else None
        if r.text:
            um = r.usage_metadata
            return dict(text=r.text, latency=time.perf_counter() - t0, model_version=r.model_version, finish=finish,
                        relaxed=relaxed, in_tok=um.prompt_token_count or 0, out_tok=um.candidates_token_count or 0,
                        think_tok=um.thoughts_token_count or 0)
        tried.append((finish, block))
    raise RuntimeError(f"empty reply after {empty_retries} tries (finish, block): {tried}")

def gemini_items(ctxs):
    items = []
    for c in ctxs.values():
        msgs = drafter_messages(c, c["evidence"])
        items.append((str(c["u"]), dict(system=msgs[0]["content"], user=msgs[1]["content"], schema=gemini_schema(c["order_full"]))))
    return items

def gemini_cost(rows):
    if CFG["gemini_usd_per_m_in"] is None:
        return None
    tin = sum(r["in_tok"] for r in rows)
    tout = sum(r["out_tok"] + r["think_tok"] for r in rows)
    return (tin * CFG["gemini_usd_per_m_in"] + tout * CFG["gemini_usd_per_m_out"]) / 1e6

GEM, t_gem = await run_requests(gemini_items(CTX), gemini_call, RES / "gemini.jsonl", 8, "gemini [shuffled]")
GEM_CF, _ = await run_requests(gemini_items(CTX_CF), gemini_call, RES / "gemini_cf.jsonl", 8, "gemini [CF order]")
for name, gem in [("shuffled", GEM), ("CF order", GEM_CF)]:
    print(f"gemini [{name}] answered {len(gem)}/{len(CTX)} users | relaxed schema: {sum(r.get('relaxed', False) for r in gem.values())}"
          f" | finish reasons: {pd.Series([r.get('finish') for r in gem.values()]).value_counts(dropna=False).to_dict()}"
          f" | model: {({r['model_version'] for r in gem.values()})} | cost USD: {gemini_cost(list(gem.values()))}")

## §10 (Optional) Teacher data for a fine-tuned drafter

This is the paper's drafter recipe. A strong model writes a rationale E for each (question, answer, documents) triple, and the drafter is fine-tuned to produce (answer, rationale) from (question, documents).

To avoid leaking test data:
- Teacher users are **not** in the test sample.
- Their history is the **train** window, and the "answer" is their **val** positives that appear among the CF candidates (val protocol).
- The test window of every user is never touched.

The output is `teacher_train.jsonl`. Save it as a Kaggle dataset named `specrag-teacher` and run the companion notebook `LLM_RS_Spec_RAG_v2_qlora.ipynb` in a fresh session; it produces the LoRA adapter that §10b loads.

In [ ]:
RUN_TEACHER = False

TEACH_TASK = ("The user actually went on to enjoy these candidates: {gold}. Write the rationale (at most 120 words) a "
              "recommender should have written, before knowing this, to rank them highly: connect the user's history "
              "and the plot evidence (cite evidence numbers like [2]) to these picks. Do not mention that you were "
              "told the outcome.")

if RUN_TEACHER:
    T = val["target_pos"]
    keep = [(u, c) for u, c in zip(TEACH_POOL, cands_teach) if set(T[u].indices) & set(c)][:CFG["n_teacher"]]
    TCTX = build_contexts([u for u, _ in keep], val, [c for _, c in keep], seed=SEED + 1)
    for c in tqdm(TCTX.values(), desc="teacher evidence"):
        add_evidence(c)

    items = []
    for c in TCTX.values():
        liked_next = set(T[c["u"]].indices)
        gold = [i for i in c["cands"] if i in liked_next][:K]
        c["answer"] = backfill(gold, c["cands"])
        c["sub_j"] = int(np.random.default_rng(c["rng_seed"]).integers(len(c["subsets"])))
        msgs = drafter_messages(c, c["subsets"][c["sub_j"]])
        user = msgs[1]["content"] + "\n\n" + TEACH_TASK.format(gold="; ".join(item_title[i] for i in gold))
        items.append((str(c["u"]), dict(system=SYSTEM, user=user, schema=gemini_schema(c["cands"], with_ranking=False))))
    TEACH, _ = await run_requests(items, gemini_call, RES / "teacher_gemini.jsonl", 8, "teacher")
    print("teacher cost USD:", gemini_cost(list(TEACH.values())))

    with (RES / "teacher_train.jsonl").open("w") as f:
        for c in TCTX.values():
            r = TEACH.get(str(c["u"]))
            if not r:
                continue
            msgs = drafter_messages(c, c["subsets"][c["sub_j"]])
            rationale = parse_draft(r["text"])[0]
            if not rationale:
                continue
            target = canonical_answer(rationale, c["answer"])
            f.write(json.dumps({"messages": msgs + [{"role": "assistant", "content": target}]}, ensure_ascii=False) + "\n")
    print("wrote", RES / "teacher_train.jsonl")

### §10b Spec RAG with the fine-tuned drafter

This runs only if the LoRA adapter from the companion notebook is attached at `CFG["lora_path"]`. The adapter was trained on a bnb-4bit copy of Qwen2.5-7B-Instruct and is served here on the AWQ copy of the same model. The small quantisation mismatch is a known approximation.

In [ ]:
SPEC_FT = single_ft = lat_ft = walls_ft = None
if Path(CFG["lora_path"]).exists():
    free_gpu()
    LORA_ARGS = DRAFTER_ARGS + ["--enable-lora", "--max-lora-rank", "16", "--lora-modules", f"drafter-ft={CFG['lora_path']}"]
    single_ft, drafts_ft, verdicts_ft, walls_ft = await run_specrag(CTX, "ft", "drafter-ft", LORA_ARGS)
    SPEC_FT = combine(CTX, drafts_ft, verdicts_ft)
    pd.to_pickle(SPEC_FT, RES / "spec_ft.pkl")
    lat_ft = await latency_probe(CTX, LAT_USERS, "drafter-ft", LORA_ARGS)
    lat_ft.to_csv(RES / "latency_ft.csv", index=False)
else:
    print("no LoRA adapter attached; skipping the fine-tuned drafter")

## §11 Grounding of the explanations

**NLI grounding.** Each rationale is split into sentences. A sentence counts as *grounded* if at least one evidence chunk shown to that system entails it with p ≥ 0.5 (DeBERTa-v3-large NLI). Sentences about the user's taste ("you liked dark thrillers") can't be entailed by a plot, so this score is a **lower bound**. Compare it between systems, not against 100%.

**Out-of-context titles.** A movie title that appears in a rationale but was not in the prompt (not a candidate, liked or disliked movie). It's matched against all 62k MovieLens titles of 3+ words, or 2 words and 12+ characters, so everyday phrases like "In Time" don't count. Run 1 flagged phrases and fragments, so two rules were added:
- In-context titles are found and masked first, longest first, so a fragment of one ("Christmas Story" inside "A Christmas Story") isn't flagged.
- A match must be written in title case ("a perfect match" isn't flagged; "A Perfect Match" is).

The examples column lists what was flagged, for checking false positives by eye. Rankings can't contain an invented title because of the schema, so rationales are where this can happen.

In [ ]:
from transformers import AutoModelForSequenceClassification

nli_tok = AutoTokenizer.from_pretrained(CFG["nli_model"])
nli = AutoModelForSequenceClassification.from_pretrained(CFG["nli_model"]).half().to(DEV).eval()
ENT = next(i for i, l in nli.config.id2label.items() if l.lower().startswith("entail"))

@torch.no_grad()
def entail_probs(pairs, bs=32):
    out = []
    for s in range(0, len(pairs), bs):
        prem, hyp = zip(*pairs[s:s + bs])
        x = nli_tok(list(prem), list(hyp), truncation="only_first", max_length=512, padding=True, return_tensors="pt").to(DEV)
        out.append(nli(**x).logits.float().softmax(-1)[:, ENT].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text or "") if len(s.split()) >= 4]

def grounding(records):
    """records: [(user, rationale, evidence chunk ids)] -> per-user share of grounded sentences."""
    pairs, owner = [], []
    for n, (_, rationale, ev) in enumerate(records):
        for si, s in enumerate(sentences(rationale)):
            pairs += [(chunk_text_[c], s) for c in ev]
            owner += [(n, si)] * len(ev)
    best = pd.DataFrame(owner, columns=["n", "s"]).assign(p=entail_probs(pairs)).groupby(["n", "s"]).p.max()
    return (best >= 0.5).groupby(level="n").mean().reindex(range(len(records))).values

def norm_text(t):
    return " ".join(re.sub(r"[^\w]+", " ", t.lower()).split())

def norm_title(t):
    return norm_text(re.sub(r"\(.*?\)|#\d+", " ", t))

# titles of 2 short words ("In Time", "The Room") are everyday phrases; require 3+ words or 12+ characters
ALL_TITLES = {norm_title(t) for t in pd.read_parquet(DATA / "all_titles.parquet").title_clean}
ALL_TITLES = {t for t in ALL_TITLES if len(t.split()) >= 3 or (len(t.split()) == 2 and len(t) >= 12)}
MAX_NGRAM = max(len(t.split()) for t in ALL_TITLES)

def title_cased(words):
    return words[0][:1].isupper() and all(w[:1].isupper() or w.isdigit() or len(w) <= 3 for w in words)

def out_of_context_titles(rationale, ctx):
    allowed = {norm_title(item_title[i]) for i in ctx["cands"] + ctx["liked"] + ctx["disliked"]} - {""}
    raw = re.sub(r"[^\w]+", " ", rationale or "").split()
    words = [w.lower() for w in raw]
    used, found = np.zeros(len(words), bool), set()
    for n in range(max([MAX_NGRAM] + [len(t.split()) for t in allowed]), 0, -1):
        for s in range(len(words) - n + 1):
            if used[s:s + n].any():
                continue
            g = " ".join(words[s:s + n])
            if g in allowed:
                used[s:s + n] = True
            elif n >= 2 and g in ALL_TITLES and title_cased(raw[s:s + n]):
                found.add(g)
                used[s:s + n] = True
    return found

def explanation_stats(name, rows):
    """rows: {user: (rationale, evidence chunk ids)}"""
    recs = [(u, r, ev) for u, (r, ev) in rows.items()]
    g = grounding(recs)
    ooc = [out_of_context_titles(r, CTX[u]) for u, r, _ in recs]
    return dict(system=name, grounded=np.nanmean(g), ooc_rate=np.mean([len(x) > 0 for x in ooc]),
                ooc_examples=sorted({t for x in ooc for t in x})[:10])

def single_rationales(single):
    return {c["u"]: (parse_draft(single[f"{c['u']}:all"]["text"])[0], c["evidence"])
            for c in CTX.values() if f"{c['u']}:all" in single}

def spec_rationales(spec):
    return {u: (v["best"]["rationale"], v["best"]["evidence"]) for u, v in spec.items() if v["best"]}

def gemini_rationales(gem):
    return {int(u): (parse_draft(r["text"])[0], CTX[int(u)]["evidence"]) for u, r in gem.items()}

EXPL = [explanation_stats("CF + drafter (single)", single_rationales(single_base)),
        explanation_stats("Spec RAG", spec_rationales(SPEC_BASE)),
        explanation_stats("CF + Gemini", gemini_rationales(GEM)),
        explanation_stats("[CF order] CF + drafter (single)", single_rationales(single_cf)),
        explanation_stats("[CF order] Spec RAG", spec_rationales(SPEC_CF)),
        explanation_stats("[CF order] CF + Gemini", gemini_rationales(GEM_CF))]
if SPEC_FT:
    EXPL += [explanation_stats("CF + drafter-FT (single)", single_rationales(single_ft)),
             explanation_stats("Spec RAG (FT drafter)", spec_rationales(SPEC_FT))]
EXPL = pd.DataFrame(EXPL)
EXPL.to_csv(RES / "explanations.csv", index=False)
display(EXPL)
del nli
free_gpu()

## §12 Results

Every row is scored on the **same 500 sampled users**, against their test-window positives.
- **ΔNDCG:** paired bootstrap difference against the candidate generator, *CF alone*. A CI that excludes 0 means the difference holds up on this sample.
- **Δ vs shuffled:** for the `[CF order]` rows, the paired difference against the same system in the shuffled arm. This is the effect of showing the CF order.
- **Gemini:** a user Gemini never answered gets the CF order; the printout below the table says how many.
- **Latency:** batch-size-1 p50/p95 per user. Gemini latency is per request and includes the network round trip.
- **Cost:** Gemini cost is from logged token usage. Local systems show GPU-seconds per 1,000 users (batched wall time × 2 GPUs). Latency was measured on the shuffled arm and is shown for both.

In [ ]:
def ranked(lists):
    return np.array([lists[int(u)] for u in SAMPLE])

def single_lists(single):
    return {c["u"]: backfill(parse_draft(single[f"{c['u']}:all"]["text"])[1], c["cands"]) if f"{c['u']}:all" in single
            else c["cands"][:K] for c in CTX.values()}

def gemini_lists(gem):
    return {c["u"]: backfill(parse_draft(gem[str(c["u"])]["text"])[1] if str(c["u"]) in gem else [], c["cands"])
            for c in CTX.values()}

def json_stats(records):
    st = [parse_draft(r["text"])[2] for r in records]
    return np.mean([s["valid"] for s in st]), np.mean([s["dups"] for s in st])

SYSTEMS = {f"{n} (all users)": None for n in GRID}
SYSTEMS.update({n: {int(u): r for u, r in cf_recs[n].items()} for n in GRID})
cand_lists = {c["u"]: c["cands"][:K] for c in CTX.values()}
SYSTEMS["CF alone (candidate order)"] = cand_lists
ARMS = {"": (GEM, single_base, SPEC_BASE), "[CF order] ": (GEM_CF, single_cf, SPEC_CF)}
for arm, (gem, single, spec) in ARMS.items():
    SYSTEMS[f"{arm}CF + Gemini (single call)"] = gemini_lists(gem)
    SYSTEMS[f"{arm}CF + drafter (single)"] = single_lists(single)
    SYSTEMS[f"{arm}Spec RAG (argmax, paper)"] = {u: v["argmax"] for u, v in spec.items()}
    SYSTEMS[f"{arm}Spec RAG (Borda)"] = {u: v["borda"] for u, v in spec.items()}
    SYSTEMS[f"{arm}Spec RAG + RRF with CF"] = {u: v["rrf"] for u, v in spec.items()}
if SPEC_FT:
    SYSTEMS["CF + drafter-FT (single)"] = single_lists(single_ft)
    SYSTEMS["Spec RAG, FT drafter (argmax)"] = {u: v["argmax"] for u, v in SPEC_FT.items()}
    SYSTEMS["Spec RAG, FT drafter + RRF"] = {u: v["rrf"] for u, v in SPEC_FT.items()}

lat = {"CF + Gemini (single call)": np.array([r["latency"] for r in GEM.values()]),
       "CF + drafter (single)": lat_base.single.values, "Spec RAG": lat_base.specrag.values,
       "[CF order] CF + Gemini (single call)": np.array([r["latency"] for r in GEM_CF.values()]),
       "[CF order] CF + drafter (single)": lat_base.single.values, "[CF order] Spec RAG": lat_base.specrag.values}
if lat_ft is not None:
    lat.update({"CF + drafter-FT (single)": lat_ft.single.values, "Spec RAG, FT drafter": lat_ft.specrag.values})
per_1k = 1000 / len(SAMPLE)
gpu_s = {"CF + drafter (single)": walls_base["single"] * 2 * per_1k,
         "Spec RAG": (walls_base["drafts"] + walls_base["verify"]) * 2 * per_1k,
         "[CF order] CF + drafter (single)": walls_cf["single"] * 2 * per_1k,
         "[CF order] Spec RAG": (walls_cf["drafts"] + walls_cf["verify"]) * 2 * per_1k}
if walls_ft:
    gpu_s.update({"CF + drafter-FT (single)": walls_ft["single"] * 2 * per_1k,
                  "Spec RAG, FT drafter": (walls_ft["drafts"] + walls_ft["verify"]) * 2 * per_1k})

base = per_user_metrics(ranked(cand_lists), SAMPLE, test["target_pos"])["ndcg"]
user_ndcg = {}
rows = []
for name, lists in SYSTEMS.items():
    if lists is None:
        r = cf_table[cf_table.model == name.split(" (")[0]].iloc[0]
        rows.append(dict(system=name, n=r.n, **{"NDCG@10": fmt(r["ndcg@10"]), "Recall@10": fmt(r["recall@10"])},
                         coverage=r.coverage, ARP=r.arp, tail_share=r.tail_share))
        continue
    recs = ranked(lists)
    m = per_user_metrics(recs, SAMPLE, test["target_pos"])
    user_ndcg[name] = m["ndcg"]
    family = max((k for k in lat if name.startswith(k)), key=len, default=None)
    row = dict(system=name, n=len(SAMPLE), **{"NDCG@10": fmt(boot_ci(m["ndcg"])), "Recall@10": fmt(boot_ci(m["recall"])),
               "ΔNDCG vs CF": fmt(boot_ci(m["ndcg"] - base))}, **list_metrics(recs, test["pop"]))
    shuffled = name.removeprefix("[CF order] ")
    if shuffled != name and shuffled in user_ndcg:
        row["Δ vs shuffled"] = fmt(boot_ci(m["ndcg"] - user_ndcg[shuffled]))
    if family:
        row.update({"p50 s": np.percentile(lat[family], 50), "p95 s": np.percentile(lat[family], 95)})
    if family in gpu_s:
        row["GPU-s / 1k users"] = gpu_s[family]
    rows.append(row)

TABLE = pd.DataFrame(rows)
for arm, (gem, _, _) in ARMS.items():
    cost = gemini_cost(list(gem.values()))
    TABLE.loc[TABLE.system == f"{arm}CF + Gemini (single call)", "API $ / 1k users"] = cost * per_1k if cost is not None else np.nan
TABLE.to_csv(RES / "ablation.csv", index=False)
pd.set_option("display.max_colwidth", 60)
display(TABLE.round(4))

for arm, (gem, _, spec), drafts in zip(ARMS, ARMS.values(), [drafts_base, drafts_cf]):
    valid_d, dups_d = json_stats(drafts.values())
    valid_g, dups_g = json_stats(gem.values())
    print(f"{arm or '[shuffled] '}drafter JSON valid {valid_d:.1%}, mean duplicate titles {dups_d:.2f} | "
          f"gemini answered {len(gem)}/{len(SAMPLE)}, JSON valid {valid_g:.1%}, duplicates {dups_g:.2f} | "
          f"Spec RAG fallback users: {sum(v['fallback'] for v in spec.values())}")

### Where do the picks come from?

For each LLM system:
- **first 10 shown:** share of its top 10 taken from the first 10 candidates listed. Without position bias that's 20% in the shuffled arm. In the CF-order arm, the first 10 listed are the CF top 10.
- **in CF top 10:** share of its picks that are also CF's top 10.
- **NDCG, CF-reordered:** its own 10 picks sorted back into CF order. If this beats the system's NDCG, the model chose a good set but ordered it badly.

In [ ]:
def pick_sources(lists, ctxs):
    shown, in_cf = [], []
    for u in SAMPLE:
        c = ctxs[int(u)]
        first = set(c["order_full"][:K])
        shown.append(np.mean([i in first for i in lists[int(u)]]))
        in_cf.append(np.mean([i in set(c["cands"][:K]) for i in lists[int(u)]]))
    reordered = {int(u): sorted(lists[int(u)], key=CTX[int(u)]["cands"].index) for u in SAMPLE}
    return {"first 10 shown": np.mean(shown), "in CF top 10": np.mean(in_cf),
            "NDCG": per_user_metrics(ranked(lists), SAMPLE, test["target_pos"])["ndcg"].mean(),
            "NDCG, CF-reordered": per_user_metrics(ranked(reordered), SAMPLE, test["target_pos"])["ndcg"].mean()}

PICKS = pd.DataFrame({name: pick_sources(SYSTEMS[name], CTX_CF if name.startswith("[CF order]") else CTX)
                      for name in SYSTEMS if any(k in name for k in ("Gemini", "drafter", "Spec RAG"))}).T
PICKS.to_csv(RES / "pick_sources.csv")
display(PICKS.round(4))

### Which score drives the choice?

These checks ask whether the verifier adds anything:
- How often does the argmax draft differ from the draft with the best ρ_Draft alone?
- Does the winning draft's NDCG beat the average draft's?

If the answers are "rarely" and "no", the verifier isn't earning its GPU.

In [ ]:
def draft_level(spec):
    rows = []
    for u, v in spec.items():
        if len(v["scored"]) < 2:
            continue
        nd = [per_user_metrics(np.array([backfill(s["ranking"], CTX[u]["cands"])]), np.array([u]), test["target_pos"])["ndcg"][0]
              for s in v["scored"]]
        parts = np.array([s["parts"] for s in v["scored"]])
        total = parts.sum(1)
        rows.append(dict(u=u, chosen=nd[int(np.argmax(total))], by_draft_only=nd[int(np.argmax(parts[:, 0]))],
                         mean_draft=np.mean(nd), oracle=max(nd), verifier_changed=np.argmax(total) != np.argmax(parts[:, 0])))
    return pd.DataFrame(rows)

for arm, spec in [("shuffled", SPEC_BASE), ("CF order", SPEC_CF)]:
    DL = draft_level(spec)
    DL.to_csv(RES / f"draft_level_{arm.replace(' ', '_')}.csv", index=False)
    print(f"[{arm}] verifier changed the pick for {DL.verifier_changed.mean():.1%} of users | mean NDCG of the draft picked by:")
    print(DL[["chosen", "by_draft_only", "mean_draft", "oracle"]].mean().round(4).to_string(), "\n")

### Failure cases

These are the users where Spec RAG lost most against CF alone in each arm, with the full trail: history, the chosen draft's rationale and scores, and what the user actually liked next.

In [ ]:
for arm, spec in [("", SPEC_BASE), ("[CF order] ", SPEC_CF)]:
    spec_nd = user_ndcg[f"{arm}Spec RAG (argmax, paper)"]
    print(f"===== {arm or '[shuffled] '}arm")
    for idx in np.argsort(spec_nd - base)[:3]:
        u = int(SAMPLE[idx])
        c, v = CTX[u], spec[u]
        print(f"user {u}: NDCG Spec RAG {spec_nd[idx]:.3f} vs CF {base[idx]:.3f}")
        print("  liked:", "; ".join(item_title[i] for i in c["liked"][:8]))
        print("  actually liked next:", "; ".join(item_title[i] for i in test["target_pos"][u].indices))
        print("  CF top-10:", "; ".join(item_title[i] for i in c["cands"][:K]))
        print("  Spec RAG:", "; ".join(item_title[i] for i in v["argmax"]))
        if v["best"]:
            print("  rationale:", v["best"]["rationale"])
            print("  draft scores (log ρ_Draft, log ρ_SC, log ρ_SR):", [tuple(round(x, 2) for x in s["parts"]) for s in v["scored"]])
        print()